In [ ]:
# Run this once if your notebook environment is missing the project libraries.
%pip install pandas numpy pillow matplotlib opencv-python scikit-image scikit-learn tqdm

# 01 - Data Prep

This notebook does the first major job of the project: it turns the raw SD302g files into a clean labelled dataset.

The fingerprint images and the labels are not stored in one simple table. The image files live in the SD302 image folders, while the pattern labels are hidden inside the SD302g `.irr` records as coded EBTS fields. This notebook connects those two sides together.

By the end, we should have:

- a table linking each usable fingerprint image to its pattern label,
- a separate table that preserves repeated labels from the raw records,
- class-count summaries for modelling,
- a small labelled image pack for manual review.

The cells are kept short on purpose so the process is easy to inspect and explain.

In [ ]:
from pathlib import Path
import re
from typing import Optional

import pandas as pd

## Project paths

First we define the folders the notebook will use.

- `EBTS_ROOT` points to the raw `.irr` files.
- `OUTPUT_DIR` is where we save clean CSV files and review images.

The small check at the end tells us whether the notebook is looking in the right place before we start parsing thousands of files.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EBTS_ROOT = PROJECT_ROOT / "ebts"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("EBTS folder exists:", EBTS_ROOT.exists())
print("Output folder:", OUTPUT_DIR)

## ANSI/NIST separators

The `.irr` records are not normal CSV or JSON files. They are EBTS/ANSI-NIST records, so they use invisible control characters to separate fields and repeated values.

We name those separator characters here so the parser can read fields correctly. This is important because one fingerprint record can contain more than one pattern entry in `9.307`.

In [ ]:
FS = "\x1c"  # file separator
GS = "\x1d"  # field boundary
RS = "\x1e"  # repeated entries
US = "\x1f"  # subfields

## Helper functions

These helpers do the low-level parsing work.

- `read_irr_text` opens the raw `.irr` file while keeping the control characters intact.
- `get_field` extracts a specific EBTS field, such as `9.307` for pattern labels.
- `split_repeated_field` handles cases where one field contains several repeated entries.
- `label_from_parts` turns codes like `WU` and `PW` into a readable combined label like `WU+PW`.

Keeping these helpers small makes it easier to trust the label extraction.

In [ ]:
def read_irr_text(path: Path) -> str:
    """Read an IRR file while preserving control characters."""
    return path.read_text(encoding="latin-1", errors="ignore")


def get_field(text: str, tag: str) -> Optional[str]:
    """Return the raw value for a field like 9.307 or 9.331."""
    match = re.search(rf"{re.escape(tag)}:([^{GS}{FS}]*)", text)
    return match.group(1) if match else None


def split_repeated_field(value: Optional[str]) -> list[list[str]]:
    """Split a repeated EBTS field into lists of subfield values."""
    if not value:
        return []

    entries = []
    for entry in value.split(RS):
        parts = [part.strip() for part in entry.split(US) if part.strip()]
        if parts:
            entries.append(parts)
    return entries


def label_from_parts(parts: list[str]) -> str:
    """Convert ['WU', 'PW'] into 'WU+PW'."""
    return "+".join(parts)

## Parse filename metadata

The filename gives us useful metadata before we even open the image.

For example, a file like `00002306_U_1000_roll_06.irr` tells us:

- subject ID: `00002306`,
- device: `U`,
- resolution: `1000`,
- capture type: `roll`,
- finger position: `06`.

We parse this information so every label can later be traced back to the subject, finger, and capture device.

In [ ]:
IRR_NAME_PATTERN = re.compile(
    r"^(?P<subject_id>\d{8})_"
    r"(?P<device>[A-Z])_"
    r"(?P<resolution>\d+)_"
    r"(?P<capture_type>roll|slap|plain)_"
    r"(?P<finger_position>\d{2})\.irr$"
)


def parse_filename(path: Path) -> dict:
    match = IRR_NAME_PATTERN.match(path.name)
    return match.groupdict() if match else {}


def collection_from_path(path: Path) -> str:
    parts = {part.lower() for part in path.parts}
    if "baseline" in parts:
        return "baseline"
    if "challengers" in parts:
        return "challengers"
    return "unknown"

## Label meanings

Field `9.307` is the key classification field. It stores the fingerprint pattern as a short code rather than as full text.

Broad labels:

- `AU` = arch
- `LS` = left-slant loop
- `RS` = right-slant loop
- `WU` = whorl
- `UC` = unclassifiable

Subtype labels appear as combined labels:

- `AU+PA` = plain arch
- `AU+TA` = tented arch
- `WU+PW` = plain whorl
- `WU+CP` = central pocket loop whorl
- `WU+DL` = double loop whorl
- `WU+AW` = accidental whorl

We keep both the broad class and the subtype because the broad classifier will be stronger, while subtype classification is useful but more imbalanced.

In [ ]:
BROAD_CLASS_MAP = {
    "AU": "arch",
    "LS": "left_slant_loop",
    "RS": "right_slant_loop",
    "WU": "whorl",
    "UC": "unclassifiable",
}

SUBTYPE_MAP = {
    "AU+PA": "plain_arch",
    "AU+TA": "tented_arch",
    "WU+PW": "plain_whorl",
    "WU+CP": "central_pocket_loop_whorl",
    "WU+DL": "double_loop_whorl",
    "WU+AW": "accidental_whorl",
}


def broad_class(label: Optional[str]) -> Optional[str]:
    if not label:
        return None
    primary_code = label.split("+")[0]
    return BROAD_CLASS_MAP.get(primary_code, "unknown")


def subtype_name(label: Optional[str]) -> Optional[str]:
    if not label:
        return None
    return SUBTYPE_MAP.get(label, BROAD_CLASS_MAP.get(label, "unknown_subtype"))

## Match IRR records to PNG images

This is the most important data-prep step.

The `.irr` record gives us the label, but the classifier needs the actual PNG image. So for each IRR record, we build the expected PNG path from the metadata.

The matching rules are slightly different by collection:

- challenger rolled images are stored under `sd302a`,
- baseline rolled and slap images are stored under `sd302b`,
- slap fingers from positions `01-10` often use the `slap-segmented` folder.

If the PNG exists, we save its path. If not, we mark the row as unmatched so it does not silently enter training.

In [ ]:
def find_matching_png(row: dict) -> Optional[Path]:
    subject = row["subject_id"]
    device = row["device"]
    resolution = row["resolution"]
    capture = row["capture_type"]
    finger = row["finger_position"]
    collection = row["collection_type"]

    if collection == "challengers":
        candidate = PROJECT_ROOT / "sd302a" / "images" / "challengers" / device / "roll" / "png" / f"{subject}_{device}_roll_{finger}.png"
        return candidate if candidate.exists() else None

    if collection == "baseline":
        capture_folder = "slap-segmented" if capture == "slap" and int(finger) <= 10 else capture
        candidate = PROJECT_ROOT / "sd302b" / "images" / "baseline" / device / resolution / capture_folder / "png" / f"{subject}_{device}_{resolution}_{capture}_{finger}.png"
        return candidate if candidate.exists() else None

    return None

## Parse all IRR files

Now we loop through every `.irr` file and build the labelled dataset.

We intentionally create two tables:

1. `image_labels`: one row per IRR file. This is convenient for image-level training and image matching.
2. `pattern_entries`: one row per repeated `9.307` entry. This preserves the raw label structure, including multi-label records.

This separation matters because some IRR files contain more than one pattern entry. We do not want to lose that information during preprocessing.

In [ ]:
irr_files = sorted(EBTS_ROOT.rglob("*.irr"))

image_rows = []
entry_rows = []

for irr_path in irr_files:
    text = read_irr_text(irr_path)
    file_meta = parse_filename(irr_path)
    if not file_meta:
        continue

    pattern_entries_raw = split_repeated_field(get_field(text, "9.307"))
    labels = [label_from_parts(parts) for parts in pattern_entries_raw]

    row = {
        **file_meta,
        "collection_type": collection_from_path(irr_path),
        "irr_path": str(irr_path.relative_to(PROJECT_ROOT)),
        "all_pattern_labels": "|".join(labels),
        "num_pattern_labels": len(labels),
        "primary_label": labels[0] if labels else None,
    }
    row["broad_class"] = broad_class(row["primary_label"])
    row["subtype"] = subtype_name(row["primary_label"])

    png_path = find_matching_png(row)
    row["png_path"] = str(png_path.relative_to(PROJECT_ROOT)) if png_path else None
    row["png_found"] = png_path is not None
    image_rows.append(row)

    for entry_index, label in enumerate(labels, start=1):
        entry_rows.append({
            **file_meta,
            "collection_type": row["collection_type"],
            "irr_path": row["irr_path"],
            "entry_index": entry_index,
            "pattern_label": label,
            "broad_class": broad_class(label),
            "subtype": subtype_name(label),
        })

image_labels = pd.DataFrame(image_rows)
pattern_entries = pd.DataFrame(entry_rows)

print("IRR files found:", len(irr_files))
print("Image-label rows:", len(image_labels))
print("Pattern-entry rows:", len(pattern_entries))

## Count the labels

This count is our first major sanity check.

If parsing worked correctly, the repeated `9.307` entries should give the known class distribution for this workspace. These counts tell us whether the dataset is balanced enough for each modelling stage.

The broad classes have useful counts, while some subtypes are small and will need class weighting, augmentation, or careful reporting.

In [ ]:
pattern_entries["pattern_label"].value_counts()

## Check image matching

A label is only useful for model training if we can connect it to an image.

This check shows how many IRR records have a matching PNG path. Any unmatched records should be investigated or excluded from the first training dataset.

In [ ]:
image_labels["png_found"].value_counts(dropna=False)

## Save clean CSV files

We save the parsed data so later notebooks do not need to re-read every raw `.irr` file.

These CSV files become the foundation for:

- manual label review,
- train/validation/test splitting,
- model training,
- app development,
- reporting.

In [ ]:
image_labels_path = OUTPUT_DIR / "irr_image_labels.csv"
pattern_entries_path = OUTPUT_DIR / "irr_pattern_entries.csv"
class_counts_path = OUTPUT_DIR / "pattern_label_counts.csv"

image_labels.to_csv(image_labels_path, index=False)
pattern_entries.to_csv(pattern_entries_path, index=False)
pattern_entries["pattern_label"].value_counts().rename_axis("label").reset_index(name="count").to_csv(class_counts_path, index=False)

print("Saved:", image_labels_path)
print("Saved:", pattern_entries_path)
print("Saved:", class_counts_path)

## Manual review sample

The labels come from the raw EBTS field `9.307`, but a small human review is still important.

The reviewer is not labelling the full dataset from scratch. They are checking that our parser did the right thing:

- the code was extracted correctly,
- the code was attached to the right image,
- the visible fingerprint pattern roughly agrees with the label.

This builds confidence before we train a classifier on thousands of images.

In [ ]:
review_sample = (
    image_labels[image_labels["png_found"]]
    .groupby("primary_label", group_keys=False)
    .apply(lambda group: group.sample(min(len(group), 10), random_state=42))
    .reset_index(drop=True)
)

review_sample_path = OUTPUT_DIR / "manual_review_sample.csv"
review_sample.to_csv(review_sample_path, index=False)

print("Manual review sample rows:", len(review_sample))
print("Saved:", review_sample_path)

## Standalone visual review sample

This section can run by itself after the processed CSV files exist.

That is useful because the reviewer package may need to be regenerated without repeating the full IRR parsing step. The section loads the saved CSV files, samples images per label, saves labelled review images, and creates a contact sheet.

The purpose is simple: put the fingerprint and the extracted label side by side so the client can quickly confirm whether the extraction and image matching look correct.

In [ ]:
from pathlib import Path
from typing import Optional
import math

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
id_columns = {"subject_id": str, "finger_position": str, "resolution": str}
image_labels = pd.read_csv(OUTPUT_DIR / "irr_image_labels.csv", dtype=id_columns)
pattern_entries = pd.read_csv(OUTPUT_DIR / "irr_pattern_entries.csv", dtype=id_columns)

print("Loaded image rows:", len(image_labels))
print("Loaded pattern entries:", len(pattern_entries))

### Build a review table

The subtype labels and repeated entries live in `pattern_entries`. The matched PNG paths live in `image_labels`.

We merge both tables using `irr_path`, because that path identifies the original raw record. After merging, each review row has both a label and an image path.

In [ ]:
review_pool = pattern_entries.merge(
    image_labels[["irr_path", "png_path", "png_found"]],
    on="irr_path",
    how="left",
)

review_pool = review_pool[review_pool["png_found"] == True].copy()
review_pool["image_path"] = review_pool["png_path"].apply(lambda path: PROJECT_ROOT / path)

review_pool["pattern_label"].value_counts()

### Sample 5 per label

We take a small sample from every available label so the review is balanced.

This avoids a common mistake: if we sampled randomly from the whole dataset, the large loop and whorl classes would dominate, and rare labels like `WU+CP` or `WU+AW` might not appear at all.

Five per label is enough for a quick confidence check without overwhelming the reviewer.

In [ ]:
samples_per_label = 5

sampled_groups = []

for label, label_rows in review_pool.groupby("pattern_label"):
    sample_size = min(len(label_rows), samples_per_label)
    sampled_groups.append(label_rows.sample(sample_size, random_state=42))

visual_review_sample = pd.concat(sampled_groups, ignore_index=True)
visual_review_sample = visual_review_sample.sort_values(
    ["pattern_label", "subject_id", "finger_position"]
).reset_index(drop=True)

visual_review_path = OUTPUT_DIR / "manual_review_visual_sample.csv"
visual_review_sample.to_csv(visual_review_path, index=False)

print("Review rows:", len(visual_review_sample))
print("Saved:", visual_review_path)
visual_review_sample["pattern_label"].value_counts().sort_index()

### Save the review images

This creates labelled copies of the selected fingerprint images.

Each saved image has a visible banner printed on it with:

- the pattern code,
- the subtype name,
- the subject ID,
- the finger position,
- the repeated-entry index.

That makes the review package practical: the client can open the images directly and does not need to cross-check the CSV.

In [ ]:
review_image_dir = OUTPUT_DIR / "manual_review_images"
review_image_dir.mkdir(parents=True, exist_ok=True)


def save_labelled_review_image(source_path: Path, output_path: Path, row: pd.Series) -> None:
    """Save a copy of the fingerprint with its review label printed on it."""
    image = Image.open(source_path).convert("RGB")

    banner_height = 120
    labelled_image = Image.new("RGB", (image.width, image.height + banner_height), "white")
    labelled_image.paste(image, (0, banner_height))

    draw = ImageDraw.Draw(labelled_image)
    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except OSError:
        font = ImageFont.load_default()

    label_text = f"Pattern: {row['pattern_label']} | Subtype: {row['subtype']}"
    meta_text = f"Subject: {row['subject_id']} | Finger: {row['finger_position']} | Entry: {row['entry_index']}"

    draw.rectangle((0, 0, image.width, banner_height), fill="white")
    draw.text((20, 22), label_text, fill="black", font=font)
    draw.text((20, 66), meta_text, fill="black", font=font)

    labelled_image.save(output_path)


saved_image_paths = []

for row_number, row in visual_review_sample.iterrows():
    source_path = Path(row["image_path"])
    safe_label = row["pattern_label"].replace("+", "-")
    output_name = f"{row_number + 1:03d}_{safe_label}_{row['subject_id']}_finger_{row['finger_position']}.png"
    output_path = review_image_dir / output_name
    save_labelled_review_image(source_path, output_path, row)
    saved_image_paths.append(str(output_path.relative_to(PROJECT_ROOT)))

visual_review_sample["saved_review_image"] = saved_image_paths
visual_review_sample.to_csv(visual_review_path, index=False)

print("Saved review images:", len(saved_image_paths))
print("Folder:", review_image_dir)

### Show and save a larger contact sheet

The contact sheet gives a quick overview of the whole review sample in one image.

This is useful for discussion, reporting, or sending a single visual summary. The individual labelled images are still saved separately for closer inspection.

In [ ]:
def show_review_grid(rows: pd.DataFrame, columns: int = 4, save_path: Optional[Path] = None) -> None:
    total = len(rows)
    rows_needed = math.ceil(total / columns)

    fig, axes = plt.subplots(rows_needed, columns, figsize=(columns * 5.0, rows_needed * 5.4))
    axes = axes.flatten() if total > 1 else [axes]

    for axis, (_, row) in zip(axes, rows.iterrows()):
        image = Image.open(row["image_path"])
        axis.imshow(image, cmap="gray")
        axis.axis("off")
        axis.set_title(
            f"{row['pattern_label']} | {row['subtype']}\n"
            f"Subject {row['subject_id']} | Finger {row['finger_position']}",
            fontsize=12,
        )

    for axis in axes[total:]:
        axis.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("Saved contact sheet:", save_path)
    plt.show()


contact_sheet_path = OUTPUT_DIR / "manual_review_contact_sheet.png"
show_review_grid(visual_review_sample, columns=4, save_path=contact_sheet_path)

## Next step

After this notebook, the next data step is to create the model-ready dataset.

That means:

- decide how to handle multi-label records,
- create clean broad-class labels,
- create subtype labels where useful,
- split images into train, validation, and test sets,
- keep the manual review notes available for quality control.